In [25]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Veri setini yükle
CSV_FILENAME = "../datasets/drug200.csv"
df = pd.read_csv(CSV_FILENAME)
print(f"Veri başarıyla yüklendi. Boyut: {df.shape}")

Veri başarıyla yüklendi. Boyut: (200, 6)


In [14]:
# ==========================================
# 1.1 İLK KEŞİF
# ==========================================

print("\n" + "=" * 50)
print("🔍 EKSİK VERİ ANALİZİ (Missing Values)")
print("=" * 50)

# Sadece eksik verisi olan sütunları ve oranlarını hesapla
missing_count = df.isnull().sum()
missing_percent = 100 * df.isnull().mean()

missing_df = pd.DataFrame(
    {"Eksik Sayısı": missing_count, "Oran (%)": missing_percent}
)
# Sadece eksik değeri olanları filtrele ve büyükten küçüğe sırala
missing_df = missing_df[missing_df["Eksik Sayısı"] > 0].sort_values(
    by="Eksik Sayısı", ascending=False
)

if missing_df.empty:
    print("✨ Harika! Veri setinde hiç eksik değer yok.")
else:
    print(missing_df.to_string())

print("=" * 50 + "\n")


🔍 EKSİK VERİ ANALİZİ (Missing Values)
✨ Harika! Veri setinde hiç eksik değer yok.



In [15]:
# ==========================================
# 1.2 KEŞİFSEL VERİ ANALİZİ
# ==========================================
# 1. Genel Yapı ve Tipler
print("--- Bilgiler ve Tipler ---")
print(df.info())

# 2. İstatistiksel Dağılım
print("\n--- İstatistiksel Özet ---")
display(df.describe())

# 3. Eksik Değer Kontrolü
print("\n--- Eksik Değerler ---")
missing = df.isnull().sum()
print(missing[missing > 0])

# 4. İlk 5 Satır Göz Atma
print("\n--- İlk 5 Satır ---")
display(df.head())

--- Bilgiler ve Tipler ---
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          200 non-null    int64  
 1   Sex          200 non-null    int64  
 2   BP           200 non-null    int64  
 3   Cholesterol  200 non-null    int64  
 4   Na_to_K      200 non-null    float64
 5   Drug         200 non-null    str    
dtypes: float64(1), int64(4), str(1)
memory usage: 10.5 KB
None

--- İstatistiksel Özet ---


,Age,Sex,BP,Cholesterol,Na_to_K
count,200.000000,200.000000,200.000000,200.000000,200.000000
mean,44.315000,0.480000,1.065000,1.515000,16.084485
std,16.544315,0.500854,0.839224,0.501029,7.223956
min,15.000000,0.000000,0.000000,1.000000,6.269000
25%,31.000000,0.000000,0.000000,1.000000,10.445500
50%,45.000000,0.000000,1.000000,2.000000,13.936500
75%,58.000000,1.000000,2.000000,2.000000,19.380000
max,74.000000,1.000000,2.000000,2.000000,38.247000



--- Eksik Değerler ---
Series([], dtype: int64)

--- İlk 5 Satır ---


,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,1,2,2,25.355,drugY
1,47,0,0,2,13.093,drugC
2,47,0,0,2,10.114,drugC
3,28,1,1,2,7.798,drugX
4,61,1,0,2,18.043,drugY


In [17]:
# ==========================================
# 1.3 VERİ TEMİZLEME VE ÖN İŞLEME
# ==========================================

X = df.drop(columns=['Drug'])
y = df['Drug']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [29]:
# ==========================================
# 1.4 ÖZNİTELİK MÜHENDİSLİĞİ
# ==========================================

numeric_cols = (
    df.select_dtypes(include=["float64","int32"])
    .columns.tolist()
)

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "poly",
            PolynomialFeatures(
                degree=2, interaction_only=False, include_bias=False
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[("num", numeric_transformer, numeric_cols)],
    remainder="passthrough",
)

pipeline_model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("feature_selection", SelectKBest(score_func=f_classif, k=6)),
        (
            "classifier",
            DecisionTreeClassifier(
                min_samples_split=3,
                min_samples_leaf=8,
                max_depth=4,
                criterion='entropy',
                random_state=42,
            ),
        ),
    ]
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "=" * 40)
print("--- K-Fold Cross Validation Sonuçları ---")
cv_scores = cross_val_score(pipeline_model, X, y, cv=skf, scoring="accuracy")

for i, score in enumerate(cv_scores, 1):
  print(f"Fold {i}: {score:.4f}")

print(f"\nGerçek K-Fold Başarısı (Ortalama): {np.mean(cv_scores):.4f}")
print(f"Skor Sapması (Standart Sapma)    : {np.std(cv_scores):.4f}")
print("=" * 40 + "\n")


--- K-Fold Cross Validation Sonuçları ---
Fold 1: 1.0000
Fold 2: 1.0000
Fold 3: 1.0000
Fold 4: 0.9750
Fold 5: 0.9750

Gerçek K-Fold Başarısı (Ortalama): 0.9900
Skor Sapması (Standart Sapma)    : 0.0122



In [30]:
# ==========================================
# ARA BÖLÜM: İDEAL K DEĞERİNİ BULMA
# ==========================================
for k_val in [10, 15, 20]:
    temp_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('feature_selection', SelectKBest(score_func=f_classif, k=k_val)),
        ("classifier", DecisionTreeClassifier(random_state=42,
                                              criterion="entropy",
                                              max_depth=4,
                                              min_samples_split=3,
                                              min_samples_leaf=8, )),
    ])

    scores = cross_val_score(temp_pipeline, X, y, cv=skf, scoring='accuracy')
    print(f"k = {k_val} için Ortalama K-Fold Başarısı: {np.mean(scores):.4f}")

/home/gokay/PycharmProjects/FastApiProject/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:782: UserWarning: k=10 is greater than n_features=6. All the features will be returned.
  warnings.warn(
/home/gokay/PycharmProjects/FastApiProject/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:782: UserWarning: k=10 is greater than n_features=6. All the features will be returned.
  warnings.warn(
/home/gokay/PycharmProjects/FastApiProject/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:782: UserWarning: k=10 is greater than n_features=6. All the features will be returned.
  warnings.warn(
/home/gokay/PycharmProjects/FastApiProject/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:782: UserWarning: k=10 is greater than n_features=6. All the features will be returned.
  warnings.warn(
/home/gokay/PycharmProjects/FastApiProject/.venv/lib/python3.12/

k = 10 için Ortalama K-Fold Başarısı: 0.9900
k = 15 için Ortalama K-Fold Başarısı: 0.9900
k = 20 için Ortalama K-Fold Başarısı: 0.9900


/home/gokay/PycharmProjects/FastApiProject/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:782: UserWarning: k=20 is greater than n_features=6. All the features will be returned.
  warnings.warn(


In [31]:
# ==========================================
# 2. NİHAİ MODEL EĞİTİMİ VE KAYIT
# ==========================================
pipeline_model.fit(X_train, y_train)

y_train_pred = pipeline_model.predict(X_train)
y_pred = pipeline_model.predict(X_test)

print(f"Train Doğruluk Oranı: {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Model Doğruluk Oranı (Accuracy): {accuracy_score(y_test, y_pred):.4f}\n")
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred))



Train Doğruluk Oranı: 1.0000
Model Doğruluk Oranı (Accuracy): 1.0000

Sınıflandırma Raporu:
              precision    recall  f1-score   support

       drugA       1.00      1.00      1.00         6
       drugB       1.00      1.00      1.00         3
       drugC       1.00      1.00      1.00         5
       drugX       1.00      1.00      1.00        11
       drugY       1.00      1.00      1.00        15

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



In [32]:
# ==========================================
# 3. KARA KUTUYU AÇMA: Hangi Özellikler Seçildi?
# ==========================================
preprocessor = pipeline_model.named_steps["preprocessor"]
selector = pipeline_model.named_steps["feature_selection"]
classifier = pipeline_model.named_steps["classifier"]

all_feature_names = preprocessor.get_feature_names_out()
selected_mask = selector.get_support()
selected_features = all_feature_names[selected_mask]

# RandomForest için coef_ yerine feature_importances_ kullanılır
importances = classifier.feature_importances_

feature_importance = pd.DataFrame(
    {
        "Özellik (Feature)": selected_features,
        "Önem Düzeyi (Importance)": importances,
    }
)

# --- GÖRSEL TEMİZLİK VE SIRALAMA BÖLÜMÜ ---
feature_importance["Özellik (Feature)"] = feature_importance[
    "Özellik (Feature)"
].str.replace(r"^(num__|cat__|text__|remainder__)", "", regex=True)

# Önem düzeyine göre azalan şekilde sıralayalım (mutlak değer almaya gerek yok, importances hep pozitiftir)
feature_importance = feature_importance.sort_values(
    by="Önem Düzeyi (Importance)", ascending=False
)

feature_importance["Önem Düzeyi (Importance)"] = feature_importance[
    "Önem Düzeyi (Importance)"
].round(4)
# ------------------------------------------

print("\n--- Modelin Seçtiği En İyi Özellikler ve Önem Düzeyleri ---")
print(feature_importance.to_string(index=False))

print(
    "En düşük tahmin edilen olasılık:",
    pipeline_model.predict_proba(X_test)[:, 1].min(),
)


--- Modelin Seçtiği En İyi Özellikler ve Önem Düzeyleri ---
Özellik (Feature)  Önem Düzeyi (Importance)
          Na_to_K                    0.5191
               BP                    0.3100
              Age                    0.0962
      Cholesterol                    0.0746
        Na_to_K^2                    0.0000
              Sex                    0.0000
En düşük tahmin edilen olasılık: 0.0


In [33]:
# ==========================================
# 4. MODELİ KAYDETME
# ==========================================
os.makedirs("../backend/models", exist_ok=True)
joblib.dump(pipeline_model, "../backend/models/drug200_pipeline.pkl")
joblib.dump(list(X_train.columns), "../backend/models/model_columns_decision_tree.pkl")

print(
    "Pipeline modeli ve sütun isimleri başarıyla 'models/' klasörüne"
    " kaydedildi!"
)

Pipeline modeli ve sütun isimleri başarıyla 'models/' klasörüne kaydedildi!
